In [1]:
import os
import copy
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path

import torch
import timm
import torchvision.transforms as transforms
from scipy.ndimage import convolve

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import (
    StratifiedKFold, cross_val_score,
    RandomizedSearchCV, train_test_split
)
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.ensemble import VotingClassifier, RandomForestClassifier
from scipy.stats import loguniform
from xgboost import XGBClassifier

try:
    from cuml.linear_model import LogisticRegression
    from cuml.svm import SVC
    print("Using cuML (GPU) for Logistic Regression and SVM")
except ImportError:
    from sklearn.linear_model import LogisticRegression
    from sklearn.svm import SVC
    print("cuML not found, falling back to sklearn (CPU)")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Torch device: {DEVICE}")

c:\Users\MCuartero\Uni\machine-learning\fine-grained-visual-recognition\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuML not found, falling back to sklearn (CPU)
Torch device: cpu


In [2]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    print("Running on Google Colab. Setting up Drive paths...")

    drive.mount("/content/drive")

    ZIP_PATH = Path("/content/drive/MyDrive/birds.zip")
    EXTRACT_DIR = Path("/content/data")

    checkpoint_1 = EXTRACT_DIR / "train_metadata.csv"
    checkpoint_2 = EXTRACT_DIR / "birds" / "train_metadata.csv"

    if checkpoint_1.exists() or checkpoint_2.exists():
        print("Data already unzipped.")
    else:
        if ZIP_PATH.exists():
            print("Unzipping now...")
            os.system(f"unzip -q {ZIP_PATH} -d {EXTRACT_DIR}")
            print("Unzipping complete!")
        else:
            print("Could not find 'birds.zip' in the root of your Google Drive.")
else:
    print("Running locally. Skipping Google Drive & unzip setup.")

Running locally. Skipping Google Drive & unzip setup.


In [3]:
IN_COLAB = "google.colab" in sys.modules
HAS_CUDA = torch.cuda.is_available()

if HAS_CUDA:
    BATCH_SIZE = 64
    XGB_TREES = 500
    SVM_ITER = 20
elif IN_COLAB:
    BATCH_SIZE = 32
    XGB_TREES = 300
    SVM_ITER = 15
else:
    BATCH_SIZE = 32
    XGB_TREES = 200
    SVM_ITER = 10

if IN_COLAB:
    EXTRACT_DIR = Path("/content/data")
    BASE_DIR = (EXTRACT_DIR / "birds") if (EXTRACT_DIR / "birds").exists() else EXTRACT_DIR
else:
    BASE_DIR = Path("data/birds")

TRAIN_META = BASE_DIR / "train_metadata.csv"
TEST_META = BASE_DIR / "test_metadata.csv"
COLOR_HIST = BASE_DIR / "color_histogram.csv"
HOG_PCA = BASE_DIR / "hog_pca.csv"
ADDITIONAL_FEATS = BASE_DIR / "additional_features.csv"

TRAIN_VIT_CACHE = BASE_DIR / "train_vit_base_embeddings.npy"
TEST_VIT_CACHE = BASE_DIR / "test_vit_base_embeddings.npy"
TRAIN_EFFNET_CACHE = BASE_DIR / "train_efficientnet_b3_embeddings.npy"
TEST_EFFNET_CACHE = BASE_DIR / "test_efficientnet_b3_embeddings.npy"
TRAIN_RN_CACHE = BASE_DIR / "train_resnet50_embeddings.npy"
TEST_RN_CACHE = BASE_DIR / "test_resnet50_embeddings.npy"
TRAIN_ENG_CACHE = BASE_DIR / "train_engineered_features.npy"
TEST_ENG_CACHE = BASE_DIR / "test_engineered_features.npy"

submission_dir = Path("submissions-02")
submission_dir.mkdir(parents=True, exist_ok=True)

counter = 1
SUBMISSION_PATH = submission_dir / f"submission-{counter:02d}.csv"
while SUBMISSION_PATH.exists():
    counter += 1
    SUBMISSION_PATH = submission_dir / f"submission-{counter:02d}.csv"

print(f"Using device: {DEVICE}")
print(f"Submission will be saved to: {SUBMISSION_PATH}")

Using device: cpu
Submission will be saved to: submissions-02\submission-02.csv


In [4]:
print("\n--- Loading CSV features ---")

train_meta = pd.read_csv(TRAIN_META)
test_meta = pd.read_csv(TEST_META)

color = pd.read_csv(COLOR_HIST)
hog = pd.read_csv(HOG_PCA)
add = pd.read_csv(ADDITIONAL_FEATS)

def merge_features(meta, *feature_dfs):
    df = meta.copy()
    for fdf in feature_dfs:
        df = df.merge(fdf, on="image_id", how="left")
    return df

train_df = merge_features(train_meta, color, hog, add)
test_df = merge_features(test_meta, color, hog, add)

meta_cols = ["image_id", "image_path", "class_id", "class_name"]
feat_color = [c for c in color.columns if c != "image_id"]
feat_hog = [c for c in hog.columns if c != "image_id"]
feat_add = [c for c in add.columns if c != "image_id"]
csv_feature_cols = feat_color + feat_hog + feat_add

print(f"Train samples: {len(train_df)},  Test samples: {len(test_df)}")
print(f"  color_histogram: {len(feat_color)} dims")
print(f"  hog_pca:         {len(feat_hog)} dims")
print(f"  additional:      {len(feat_add)} dims")
print(f"  total CSV:       {len(csv_feature_cols)} dims")


--- Loading CSV features ---
Train samples: 417,  Test samples: 180
  color_histogram: 96 dims
  hog_pca:         100 dims
  additional:      23 dims
  total CSV:       219 dims


In [23]:
# Feature engineering

def extract_engineered(image_paths, base_dir, cache_path):
    if cache_path.exists():
        print(f"  Loading cached engineered features from {cache_path}")
        return np.load(cache_path)

    print(f"  Extracting engineered features for {len(image_paths)} images ...")

    def gabor_kernels():
        kernels = []
        for theta in [0, 45, 90, 135]:
            for sigma in [1, 3]:
                t = np.deg2rad(theta)
                kern = np.zeros((9, 9))
                for x in range(-4, 5):
                    for y in range(-4, 5):
                        xp = x * np.cos(t) + y * np.sin(t)
                        yp = -x * np.sin(t) + y * np.cos(t)
                        kern[x+4, y+4] = np.exp(-(xp**2 + yp**2) / (2 * sigma**2)) * np.cos(2 * np.pi * xp / (2 * sigma))
                    kern /= (np.abs(kern).sum() + 1e-8)
                    kernels.append(kern)
        return kernels

    kernels = gabor_kernels()
    all_feats = []

    for rel_path in image_paths:
        full_path = base_dir / rel_path
        try:
            img = Image.open(full_path).convert("RGB").resize((64, 64))
        except Exception:
            img = Image.new("RGB", (64, 64), (128, 128, 128))

        rgb = np.array(img, dtype=np.float32) / 255.0

        # HSV statistics per channel (mean, std, skew proxy)
        import colorsys
        hsv_pixels = [colorsys.rgb_to_hsv(r, g, b) for r, g, b in rgb.reshape(-1, 3)]
        hsv = np.array(hsv_pixels)
        hsv_feats = np.concatenate([hsv.mean(0), hsv.std(0), np.percentile(hsv, 25, axis=0), np.percentile(hsv, 75, axis=0)])

        # Colour ratios between channels: red/(green+eps), yellow proxy (r+g)/2 - b
        r, g, b_ch = rgb[:,:,0].mean(), rgb[:,:,1].mean(), rgb[:,:,2].mean()
        colour_ratios = np.array([
            r / (g + 1e-6),
            g / (b_ch + 1e-6),
            r / (b_ch + 1e-6),
            (r + g) / 2 - b_ch,   # yellow proxy
            b_ch - (r + g) / 2,   # blue proxy
        ])

        # Gabor texture responses on grayscale
        gray = rgb.mean(axis=2)
        gabor_feats = []
        for kern in kernels:
            resp = convolve(gray, kern)
            gabor_feats.extend([resp.mean(), resp.std()])
        gabor_feats = np.array(gabor_feats)

        # Simple LBP: centre vs 8-neighbour average difference statistics
        padded = np.pad(gray, 1, mode="reflect")
        neighbours = np.stack([
            padded[0:-2, 0:-2], padded[0:-2, 1:-1], padded[0:-2, 2:],
            padded[1:-1, 0:-2],                      padded[1:-1, 2:],
            padded[2:,   0:-2], padded[2:,   1:-1], padded[2:,   2:],
        ], axis=0)
        lbp_diff = gray[np.newaxis] - neighbours
        lbp_feats = np.concatenate([lbp_diff.mean(axis=(1, 2)), lbp_diff.std(axis=(1, 2))])

        feat = np.concatenate([hsv_feats, colour_ratios, gabor_feats, lbp_feats])
        all_feats.append(feat)

    result = np.array(all_feats, dtype=np.float32)
    np.save(cache_path, result)
    print(f"  Saved to {cache_path}")
    return result


print("\n--- Extracting engineered features ---")
train_eng = extract_engineered(train_df["image_path"].values, BASE_DIR, TRAIN_ENG_CACHE)
test_eng = extract_engineered(test_df["image_path"].values, BASE_DIR, TEST_ENG_CACHE)
print(f"Engineered feature dims: {train_eng.shape[1]}")
print(f"  HSV stats: 12, colour ratios: 5, Gabor: 16, LBP: 16")


--- Extracting engineered features ---
  Loading cached engineered features from data\birds\train_engineered_features.npy
  Loading cached engineered features from data\birds\test_engineered_features.npy
Engineered feature dims: 49
  HSV stats: 12, colour ratios: 5, Gabor: 16, LBP: 16


In [24]:
def extract_embeddings(image_paths, base_dir, cache_path, model_name, batch_size=32):
    if cache_path.exists():
        print(f"  Loading cached embeddings from {cache_path}")
        return np.load(cache_path)

    print(f"  Extracting {model_name} embeddings for {len(image_paths)} images ...")

    model = timm.create_model(model_name, pretrained=True, num_classes=0)
    model.eval().to(DEVICE)

    preprocess = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])

    all_embeddings = []
    for i in range(0, len(image_paths), batch_size):
        batch_paths = image_paths[i: i + batch_size]
        imgs = []
        for rel_path in batch_paths:
            full_path = base_dir / rel_path
            try:
                img = Image.open(full_path).convert("RGB")
            except Exception:
                img = Image.new("RGB", (128, 128), (128, 128, 128))
            imgs.append(preprocess(img))

        batch_tensor = torch.stack(imgs).to(DEVICE)
        with torch.no_grad():
            emb = model(batch_tensor).cpu().numpy()
        all_embeddings.append(emb)

        if (i // batch_size) % 10 == 0:
            print(f"    {min(i + batch_size, len(image_paths))}/{len(image_paths)}")

    embeddings = np.vstack(all_embeddings)
    np.save(cache_path, embeddings)
    print(f"  Saved to {cache_path}")
    return embeddings


print("\n--- Extracting ViT-Base embeddings ---")
train_vit = extract_embeddings(
    train_df["image_path"].values, BASE_DIR,
    TRAIN_VIT_CACHE, "vit_base_patch16_224", BATCH_SIZE
)
test_vit = extract_embeddings(
    test_df["image_path"].values, BASE_DIR,
    TEST_VIT_CACHE, "vit_base_patch16_224", BATCH_SIZE
)

print("\n--- Extracting EfficientNet-B3 embeddings ---")
train_eff = extract_embeddings(
    train_df["image_path"].values, BASE_DIR,
    TRAIN_EFFNET_CACHE, "efficientnet_b3", BATCH_SIZE
)
test_eff = extract_embeddings(
    test_df["image_path"].values, BASE_DIR,
    TEST_EFFNET_CACHE, "efficientnet_b3", BATCH_SIZE
)

print("\n--- Extracting ResNet-50 embeddings ---")
train_rn = extract_embeddings(
    train_df["image_path"].values, BASE_DIR,
    TRAIN_RN_CACHE, "resnet50", BATCH_SIZE
)
test_rn = extract_embeddings(
    test_df["image_path"].values, BASE_DIR,
    TEST_RN_CACHE, "resnet50", BATCH_SIZE
)

print(f"ViT-Base shape:        {train_vit.shape}")
print(f"EfficientNet-B3 shape: {train_eff.shape}")
print(f"ResNet-50 shape:       {train_rn.shape}")


--- Extracting ViT-Base embeddings ---
  Loading cached embeddings from data\birds\train_vit_base_embeddings.npy
  Loading cached embeddings from data\birds\test_vit_base_embeddings.npy

--- Extracting EfficientNet-B3 embeddings ---
  Loading cached embeddings from data\birds\train_efficientnet_b3_embeddings.npy
  Loading cached embeddings from data\birds\test_efficientnet_b3_embeddings.npy

--- Extracting ResNet-50 embeddings ---
  Loading cached embeddings from data\birds\train_resnet50_embeddings.npy
  Loading cached embeddings from data\birds\test_resnet50_embeddings.npy
ViT-Base shape:        (417, 768)
EfficientNet-B3 shape: (417, 1536)
ResNet-50 shape:       (417, 2048)


In [25]:
print("\n--- Feature ablation study (SVM, 5-fold CV) ---")
# Train a fixed SVM on each feature set in isolation so we can compare their
# individual contributions — this gives concrete numbers for the report.

scaler_abl = StandardScaler()

def ablation_cv(raw_feats, label, n_comp=None):
    X = np.nan_to_num(raw_feats.astype(np.float32))
    X = scaler_abl.fit_transform(X)
    if n_comp is not None and n_comp < X.shape[1]:
        X = PCA(n_components=n_comp, random_state=42).fit_transform(X)
    clf = SVC(kernel="rbf", C=10.0, gamma="scale",
              **({} if SVC.__module__.startswith("cuml") else {"random_state": 42, "probability": False}))
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(clf, X, y_train, cv=skf, scoring="accuracy", n_jobs=-1)
    print(f"  {label:<35s}: {scores.mean():.4f} ± {scores.std():.4f}")
    return scores.mean()

y_train = train_df["class_id"].values

abl_results = {}
abl_results["Color histogram"]        = ablation_cv(train_df[feat_color].values, "Color histogram")
abl_results["HOG PCA"]                = ablation_cv(train_df[feat_hog].values,   "HOG PCA")
abl_results["Additional features"]    = ablation_cv(train_df[feat_add].values,   "Additional features")
abl_results["Engineered (HSV+Gabor)"] = ablation_cv(train_eng, "Engineered (HSV+Gabor+LBP)")
abl_results["All CSV + engineered"]   = ablation_cv(
    np.hstack([train_df[csv_feature_cols].values, train_eng]),
    "All CSV + engineered"
)
abl_results["ViT-Base only"]          = ablation_cv(train_vit, "ViT-Base only", n_comp=256)
abl_results["EfficientNet-B3 only"]   = ablation_cv(train_eff, "EfficientNet-B3 only", n_comp=128)
abl_results["ResNet-50 only"]         = ablation_cv(train_rn,  "ResNet-50 only", n_comp=128)
abl_results["All neural (combined)"]  = ablation_cv(
    np.hstack([train_vit, train_eff, train_rn]),
    "All neural (combined)", n_comp=256
)

print("\n  Ranking:")
for name, score in sorted(abl_results.items(), key=lambda x: -x[1]):
    print(f"    {name:<35s}: {score:.4f}")


--- Feature ablation study (SVM, 5-fold CV) ---
  Color histogram                    : 0.1823 ± 0.0210


  HOG PCA                            : 0.2156 ± 0.0398
  Additional features                : 0.2065 ± 0.0567
  Engineered (HSV+Gabor+LBP)         : 0.2135 ± 0.0357
  All CSV + engineered               : 0.2638 ± 0.0466
  ViT-Base only                      : 0.9329 ± 0.0234
  EfficientNet-B3 only               : 0.8967 ± 0.0323
  ResNet-50 only                     : 0.8034 ± 0.0381
  All neural (combined)              : 0.8993 ± 0.0236

  Ranking:
    ViT-Base only                      : 0.9329
    All neural (combined)              : 0.8993
    EfficientNet-B3 only               : 0.8967
    ResNet-50 only                     : 0.8034
    All CSV + engineered               : 0.2638
    HOG PCA                            : 0.2156
    Engineered (HSV+Gabor)             : 0.2135
    Additional features                : 0.2065
    Color histogram                    : 0.1823


In [26]:
print("\n--- Feature importance (Random Forest on CSV + engineered features) ---")
# Fit RF on the non-neural features to see which hand-crafted features matter most.
# Gives something concrete to discuss in the report.

X_hand = np.nan_to_num(
    np.hstack([train_df[csv_feature_cols].values, train_eng]).astype(np.float32)
)
feat_names = (
    csv_feature_cols
    + [f"hsv_{i}" for i in range(12)]
    + [f"colratio_{i}" for i in range(5)]
    + [f"gabor_{i}" for i in range(16)]
    + [f"lbp_{i}" for i in range(16)]
)

rf_imp = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_imp.fit(X_hand, y_train)

importances = pd.Series(rf_imp.feature_importances_, index=feat_names)
top20 = importances.nlargest(20)

plt.figure(figsize=(10, 6))
top20.sort_values().plot(kind="barh", color="steelblue")
plt.title("Top 20 feature importances (RF) — CSV + engineered features")
plt.xlabel("Importance")
plt.tight_layout()
output_dir = BASE_DIR / "figures"
output_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(output_dir / "feature_importance.png", dpi=150)
plt.show()
print(f"Saved: {output_dir / 'feature_importance.png'}")
print("\nTop 10 features:")
print(top20.head(10).to_string())


--- Feature importance (Random Forest on CSV + engineered features) ---
Saved: data\birds\figures\feature_importance.png

Top 10 features:
hsv_4         0.012248
hog_pca_0     0.010476
colratio_1    0.008484
colratio_4    0.008463
color_2       0.008016
color_34      0.007674
color_1       0.007655
hsv_0         0.006984
colratio_0    0.006449
color_3       0.006430


In [27]:
print("\n--- Combining and normalising all features ---")

X_train_csv = np.nan_to_num(train_df[csv_feature_cols].values.astype(np.float32))
X_test_csv = np.nan_to_num(test_df[csv_feature_cols].values.astype(np.float32))
scaler_csv = StandardScaler()
X_train_csv = scaler_csv.fit_transform(X_train_csv)
X_test_csv = scaler_csv.transform(X_test_csv)

scaler_eng = StandardScaler()
X_train_eng = scaler_eng.fit_transform(np.nan_to_num(train_eng.astype(np.float32)))
X_test_eng = scaler_eng.transform(np.nan_to_num(test_eng.astype(np.float32)))

scaler_vit = StandardScaler()
X_train_vit = scaler_vit.fit_transform(train_vit)
X_test_vit = scaler_vit.transform(test_vit)

scaler_eff = StandardScaler()
X_train_eff = scaler_eff.fit_transform(train_eff)
X_test_eff = scaler_eff.transform(test_eff)

scaler_rn = StandardScaler()
X_train_rn = scaler_rn.fit_transform(train_rn)
X_test_rn = scaler_rn.transform(test_rn)

N_TRAIN = len(train_df)
VIT_COMPS = min(256, N_TRAIN - 1)
EFF_COMPS = min(128, N_TRAIN - 1)
RN_COMPS = min(128, N_TRAIN - 1)

pca_vit = PCA(n_components=VIT_COMPS, random_state=42)
X_train_vit = pca_vit.fit_transform(X_train_vit)
X_test_vit = pca_vit.transform(X_test_vit)

pca_eff = PCA(n_components=EFF_COMPS, random_state=42)
X_train_eff = pca_eff.fit_transform(X_train_eff)
X_test_eff = pca_eff.transform(X_test_eff)

pca_rn = PCA(n_components=RN_COMPS, random_state=42)
X_train_rn = pca_rn.fit_transform(X_train_rn)
X_test_rn = pca_rn.transform(X_test_rn)

X_train = np.hstack([X_train_vit, X_train_eff, X_train_rn, X_train_csv, X_train_eng])
X_test = np.hstack([X_test_vit, X_test_eff, X_test_rn, X_test_csv, X_test_eng])

print(f"Total feature dims: {X_train.shape[1]}")
print(f"  ViT-Base (PCA):        {X_train_vit.shape[1]}")
print(f"  EfficientNet-B3 (PCA): {X_train_eff.shape[1]}")
print(f"  ResNet-50 (PCA):       {X_train_rn.shape[1]}")
print(f"  CSV:                   {X_train_csv.shape[1]}")
print(f"  Engineered:            {X_train_eng.shape[1]}")


--- Combining and normalising all features ---
Total feature dims: 780
  ViT-Base (PCA):        256
  EfficientNet-B3 (PCA): 128
  ResNet-50 (PCA):       128
  CSV:                   219
  Engineered:            49


In [28]:
def cv_evaluate(model, X, y, n_splits=5, label="Model"):
    """Stratified 5-fold cross-validation."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    scores = cross_val_score(model, X, y, cv=skf, scoring="accuracy", n_jobs=-1)
    print(f"  {label}: {scores.mean():.4f} ± {scores.std():.4f}  (folds: {np.round(scores, 4)})")
    return scores.mean()

In [29]:
print("\n--- Model 1: Logistic Regression ---")

is_cuml = LogisticRegression.__module__.startswith("cuml")
solver_name = "qn" if is_cuml else "lbfgs"

lr = LogisticRegression(
    C=5.0,
    max_iter=3000,
    solver=solver_name,
    class_weight="balanced"
)

lr_cv = cv_evaluate(lr, X_train, y_train, label="Logistic Regression")


--- Model 1: Logistic Regression ---
  Logistic Regression: 0.9521 ± 0.0105  (folds: [0.9405 0.9405 0.9518 0.9639 0.9639])


In [30]:
print("\n--- Model 2: SVM (RBF) with RandomizedSearchCV ---")

is_cuml = SVC.__module__.startswith("cuml")

if is_cuml:
    svc_kwargs = {"kernel": "rbf", "probability": True}
else:
    svc_kwargs = {"kernel": "rbf", "probability": False, "random_state": 42}

param_dist = {
    "C": loguniform(1, 100),
    "gamma": loguniform(1e-4, 1e-1),
}

svm_search = RandomizedSearchCV(
    SVC(**svc_kwargs),
    param_distributions=param_dist,
    n_iter=SVM_ITER,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    random_state=42,
    verbose=3
)

print(f"Starting RandomizedSearchCV ({SVM_ITER} iterations, 5 folds)...")
svm_search.fit(X_train, y_train)

print(f"  Best params: {svm_search.best_params_}")
print(f"  Best CV score: {svm_search.best_score_:.4f}")

if not is_cuml:
    svm = SVC(
        kernel="rbf",
        probability=True,
        random_state=42,
        **svm_search.best_params_
    )
    svm.fit(X_train, y_train)
else:
    svm = svm_search.best_estimator_

svm_cv = svm_search.best_score_


--- Model 2: SVM (RBF) with RandomizedSearchCV ---
Starting RandomizedSearchCV (10 iterations, 5 folds)...
Fitting 5 folds for each of 10 candidates, totalling 50 fits
  Best params: {'C': np.float64(2.05133826308745), 'gamma': np.float64(0.00029375384576328325)}
  Best CV score: 0.8897


In [31]:
print("\n--- Model 3: XGBoost ---")

xgb = XGBClassifier(
    n_estimators=XGB_TREES,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.7,
    min_child_weight=1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1,
    device=DEVICE
)

xgb_cv = cv_evaluate(xgb, X_train, y_train, label="XGBoost")


--- Model 3: XGBoost ---
  XGBoost: 0.9305 ± 0.0156  (folds: [0.9286 0.9048 0.9518 0.9277 0.9398])


In [32]:
print("\n--- Model 4: Soft Voting Ensemble ---")

is_cuml = LogisticRegression.__module__.startswith("cuml")
best_C = svm_search.best_params_["C"]
best_gamma = svm_search.best_params_["gamma"]

if is_cuml:
    lr_inst = LogisticRegression(C=5.0, max_iter=3000, solver="qn", class_weight="balanced")
    svm_inst = SVC(kernel="rbf", probability=True, C=best_C, gamma=best_gamma)
else:
    lr_inst = LogisticRegression(C=5.0, max_iter=3000, solver="lbfgs", class_weight="balanced")
    svm_inst = SVC(kernel="rbf", probability=True, C=best_C, gamma=best_gamma, random_state=42)

ensemble = VotingClassifier(
    estimators=[
        ("lr", lr_inst),
        ("svm", svm_inst),
        ("xgb", XGBClassifier(
            n_estimators=XGB_TREES,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.7,
            min_child_weight=1,
            reg_alpha=0.1,
            reg_lambda=1.0,
            eval_metric="mlogloss",
            random_state=42,
            n_jobs=-1,
            device=DEVICE
        )),
    ],
    voting="soft"
)

ensemble_cv = cv_evaluate(ensemble, X_train, y_train, label="Soft Voting Ensemble")


--- Model 4: Soft Voting Ensemble ---
  Soft Voting Ensemble: 0.9521 ± 0.0074  (folds: [0.9405 0.9524 0.9518 0.9639 0.9518])


In [33]:
print("\n--- Cross-validation summary ---")

results = {
    "Logistic Regression": lr_cv,
    "SVM RBF (tuned)": svm_cv,
    "XGBoost": xgb_cv,
    "Soft Voting Ensemble": ensemble_cv,
}
for name, score in sorted(results.items(), key=lambda x: -x[1]):
    print(f"  {name}: {score:.4f}")

best_name = max(results, key=results.get)
model_map = {
    "Logistic Regression": lr,
    "SVM RBF (tuned)": svm,
    "XGBoost": xgb,
    "Soft Voting Ensemble": ensemble,
}
best_model = model_map[best_name]
print(f"\nBest model: {best_name} ({results[best_name]:.4f})")

print(f"Fitting {best_name} on full training set...")
best_model.fit(X_train, y_train)
train_preds = best_model.predict(X_train)
print(f"Training accuracy (sanity check): {accuracy_score(y_train, train_preds):.4f}")


--- Cross-validation summary ---
  Logistic Regression: 0.9521
  Soft Voting Ensemble: 0.9521
  XGBoost: 0.9305
  SVM RBF (tuned): 0.8897

Best model: Logistic Regression (0.9521)
Fitting Logistic Regression on full training set...
Training accuracy (sanity check): 1.0000


In [34]:
print("\n--- Generating confusion matrix on 20% holdout ---")

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=42
)

eval_model = copy.deepcopy(best_model)
eval_model.fit(X_tr, y_tr)
val_preds = eval_model.predict(X_val)
val_acc = accuracy_score(y_val, val_preds)
print(f"Holdout validation accuracy: {val_acc:.4f}")

class_names = (
    train_df[["class_id", "class_name"]]
    .drop_duplicates()
    .sort_values("class_id")["class_name"]
    .tolist()
)

cm = confusion_matrix(y_val, val_preds)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.title(f"Confusion Matrix — {best_name} (val acc: {val_acc:.3f})")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

output_dir = BASE_DIR / "figures"
output_dir.mkdir(parents=True, exist_ok=True)
safe_model_name = best_name.lower().replace(" ", "_").replace("(", "").replace(")", "").replace("-", "_")
plt.savefig(output_dir / f"{safe_model_name}.png", dpi=150)
plt.show()
print(f"Saved: {output_dir / f'{safe_model_name}.png'}")

print("\nClassification Report:")
print(classification_report(y_val, val_preds, target_names=class_names))


--- Generating confusion matrix on 20% holdout ---
Holdout validation accuracy: 0.9524
Saved: data\birds\figures\logistic_regression.png

Classification Report:
                      precision    recall  f1-score   support

  American_Goldfinch       1.00      1.00      1.00         8
            Blue_Jay       1.00      1.00      1.00         8
            Cardinal       1.00      1.00      1.00         8
        Herring_Gull       0.80      0.89      0.84         9
       House_Sparrow       1.00      1.00      1.00         9
Red_winged_Blackbird       1.00      1.00      1.00         9
    Ring_billed_Gull       0.86      0.75      0.80         8
        Song_Sparrow       1.00      1.00      1.00         9
      Wilson_Warbler       1.00      0.88      0.93         8
      Yellow_Warbler       0.89      1.00      0.94         8

            accuracy                           0.95        84
           macro avg       0.95      0.95      0.95        84
        weighted avg       0.9

In [35]:
print("\n--- Generating test predictions ---")

test_preds = best_model.predict(X_test)
submission = pd.DataFrame({
    "image_id": test_meta["image_id"],
    "class_id": test_preds,
})
submission.to_csv(SUBMISSION_PATH, index=False)
print(f"Saved: {SUBMISSION_PATH}  ({len(submission)} rows)")


--- Generating test predictions ---
Saved: submissions-02\submission-02.csv  (180 rows)
